In [ ]:
# Module 1 (Notebook Cell)
# STFT / iSTFT utilities + ComplexFrameBuffer (ring buffer)
# - This cell defines STFTProcessor and ComplexFrameBuffer for streaming.
# - Designed to be copy-pasted/run as a single Jupyter notebook cell.
# - Comments follow ML standards: clear docstrings, tensor shapes, dtype hints.
#
# Notes for use in streaming:
# - For development and training set center=True to allow perfect istft reconstruction.
# - For low-latency streaming set center=False and manage padding/windowing externally.
#
# Run this cell in a Python notebook (requires PyTorch).

import torch
import torch.nn.functional as F
from typing import Tuple

# ----------------------------- STFT Processor -----------------------------
class STFTProcessor:
    """
    STFT / iSTFT helper class for streaming pipelines.

    Responsibilities:
    - Consistent STFT/iSTFT behavior for training and inference.
    - Provide real+imag channel format for compatibility with NN modules.
    - Precompute window to avoid repeated allocations.

    Key args:
      sample_rate: int (Hz) -- informational
      n_fft: int -- FFT size
      hop_length: int -- hop size in samples
      win_length: int -- window size in samples (defaults to n_fft)
      window_fn: callable -- returns a window tensor (e.g., torch.hann_window)
      device, dtype: torch device/dtype for window allocation
      center: bool -- whether to use centered STFT (center=True helpful for training reconstruction)
    """
    def __init__(
        self,
        sample_rate: int = 16000,
        n_fft: int = 320,
        hop_length: int = 160,
        win_length: int = None,
        window_fn=torch.hann_window,
        device: torch.device = torch.device("cpu"),
        dtype: torch.dtype = torch.float32,
        center: bool = True,
    ):
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length if win_length is not None else n_fft
        self.device = device
        self.dtype = dtype
        self.center = center

        # Persistent analysis window allocated once to avoid reallocation cost.
        self.window = window_fn(self.win_length, dtype=self.dtype, device=self.device)
        # Number of one-sided frequency bins produced by torch.stft
        self.freq_bins = self.n_fft // 2 + 1

    def stft(self, waveform: torch.Tensor) -> torch.Tensor:
        """
        Compute STFT and return complex spectrogram as real+imag channels.

        Args:
            waveform: Tensor [B, samples], float32
        Returns:
            complex_spec: Tensor [B, F, T, 2] (real, imag)
            where F = n_fft//2 + 1, T = num frames
        """
        assert waveform.dim() == 2, "waveform must be [B, samples]"
        # Use return_complex=True for modern PyTorch then split into real/imag to keep consistent format.
        st = torch.stft(
            waveform,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window,
            return_complex=True,
            center=self.center,
        )
        real = st.real
        imag = st.imag
        complex_spec = torch.stack([real, imag], dim=-1)  # [B, F, T, 2]
        return complex_spec

    def istft(self, complex_spec: torch.Tensor, length: int = None) -> torch.Tensor:
        """
        Inverse STFT from complex_spec shape [B, F, T, 2] -> waveform [B, samples].

        Args:
            complex_spec: Tensor [B, F, T, 2] (real, imag)
            length: optional int, desired output length in samples
        Returns:
            waveform: Tensor [B, samples]
        """
        assert complex_spec.dim() == 4 and complex_spec.size(-1) == 2, "complex_spec must be [B,F,T,2]"
        real = complex_spec[..., 0]
        imag = complex_spec[..., 1]
        st_complex = torch.complex(real, imag)  # [B, F, T]
        waveform = torch.istft(
            st_complex,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window,
            center=self.center,
            length=length,
        )
        return waveform

    def magnitude(self, complex_spec: torch.Tensor) -> torch.Tensor:
        """
        Compute magnitude spectrogram from complex_spec [B,F,T,2] -> [B,F,T]
        """
        real = complex_spec[..., 0]
        imag = complex_spec[..., 1]
        mag = torch.sqrt(real**2 + imag**2 + 1e-12)
        return mag

    def to_device(self, device: torch.device):
        """
        Move internal window to `device`. Note: other tensors used in pipeline must also be moved.
        """
        self.device = device
        self.window = self.window.to(device=self.device)
        self.dtype = self.window.dtype


# ----------------------------- ComplexFrameBuffer -------------------------
class ComplexFrameBuffer:
    """
    Ring buffer for storing the last K complex STFT frames for streaming deep-filter.

    Internal format: Tensor [B, K, F, 2] where last dim is (real, imag).
    Index 0 = current frame X(f,t); index 1 = X(f,t-1); ... index K-1 = oldest.
    Methods:
      - reset(batch_size)
      - append(frame_complex)  # frame_complex: [B, F, 2]
      - get_buffer() -> [B, K, F, 2]
    """
    def __init__(self, K: int, freq_bins: int, device: torch.device = torch.device("cpu"), dtype=torch.float32):
        self.K = K
        self.freq_bins = freq_bins
        self.device = device
        self.dtype = dtype
        self.buffer = None
        self.initialized = False

    def reset(self, batch_size: int = 1):
        """
        Initialize the buffer to zeros for specified batch size.
        """
        self.buffer = torch.zeros(batch_size, self.K, self.freq_bins, 2, device=self.device, dtype=self.dtype)
        self.initialized = True

    def append(self, frame_complex: torch.Tensor) -> torch.Tensor:
        """
        Append new complex frame(s) to buffer.

        Args:
            frame_complex: [B, F, 2] tensor
        Returns:
            buffer: [B, K, F, 2] after append (index 0 is the new frame)
        """
        assert frame_complex.dim() == 3 and frame_complex.size(-1) == 2, "frame_complex must be [B,F,2]"
        B, F, _ = frame_complex.shape
        if not self.initialized or self.buffer.size(0) != B:
            self.reset(batch_size=B)
        # Shift right by 1 along time-axis (dim=1); newest frame goes to index 0.
        self.buffer = torch.roll(self.buffer, shifts=1, dims=1)
        self.buffer[:, 0, :, :] = frame_complex
        return self.buffer

    def get_buffer(self) -> torch.Tensor:
        """
        Return current buffer [B, K, F, 2].
        """
        assert self.initialized, "Buffer not initialized; call reset() or append() first."
        return self.buffer


# ----------------------------- Smoke test (cell execution) -----------------
if __name__ == "__main__":
    # Quick sanity tests to validate shapes and reconstruction.
    device = torch.device("cpu")
    sr = 16000
    n_fft = 320
    hop = 160
    win = 320
    batch = 2
    duration_sec = 0.5
    samples = int(sr * duration_sec)

    # Create STFT processor (center=True gives perfect istft for testing)
    stft_proc = STFTProcessor(sample_rate=sr, n_fft=n_fft, hop_length=hop, win_length=win, device=device, center=True)
    # Random waveform batch for test
    wav = torch.randn(batch, samples, device=device)

    # Compute complex spectrogram [B, F, T, 2]
    cspec = stft_proc.stft(wav)
    print("Complex spec shape:", cspec.shape)  # expected [B, F, T, 2]

    # Magnitude spectrogram [B, F, T]
    mag = stft_proc.magnitude(cspec)
    print("Magnitude shape:", mag.shape)

    # Reconstruct waveform via istft (should be near-zero MSE)
    rec = stft_proc.istft(cspec, length=samples)
    print("Reconstructed waveform shape:", rec.shape)
    mse = torch.mean((wav - rec) ** 2).item()
    print(f"Reconstruction MSE: {mse:.6e}")

    # Test ComplexFrameBuffer append/get_buffer
    B = 1
    K = 3
    freq_bins = n_fft // 2 + 1
    buf = ComplexFrameBuffer(K=K, freq_bins=freq_bins, device=device)
    _, F, T, _ = cspec.shape
    for t in range(min(5, T)):
        frame = cspec[0, :, t, :].unsqueeze(0)  # [1, F, 2]
        b = buf.append(frame)
        print(f"After append {t+1}, buffer shape: {b.shape} -- buffer[0,0,0,:]={b[0,0,0,:].cpu().numpy()}")

    print("Module 1 (STFT + ComplexFrameBuffer) cell executed successfully.")

In [ ]:
# ============================================================
# Module 2 (Notebook Cell)
# MobileDeepFilterNet Core Architecture
# ============================================================
# This module defines the neural network used for real-time
# speech enhancement based on the MobileDeepFilterNet design.
#
# Architecture Overview:
#
# magnitude patch [B, K_ctx, F]
#          │
#          ▼
#   Mobile-style Encoder (Depthwise Separable Conv)
#          │
#          ▼
#   Frequency pooling
#          │
#          ▼
#   GRU (temporal modeling)
#          │
#          ▼
#   Decoder heads
#      ├── Spectral Mask
#      └── Deep Filter Taps
#
# Outputs:
#   mask : [B, F]
#   taps : [B, F, K_tap, 2]  (complex filter taps)

import torch
import torch.nn as nn


# ------------------------------------------------------------
# Depthwise Separable Convolution Block
# ------------------------------------------------------------
class DepthwiseSeparableConv(nn.Module):
    """
    Depthwise separable convolution block used in mobile architectures.

    Structure:
        Depthwise Conv2D
        → Pointwise Conv2D (1x1)
        → BatchNorm
        → ReLU activation

    Advantages:
        - Reduces parameter count
        - Lower MAC operations
        - Suitable for edge / mobile inference

    Input shape:
        [B, C, T, F]

    Output shape:
        [B, C_out, T, F]
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=3,
            padding=1,
            groups=in_channels,
            bias=False
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):

        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.act(x)

        return x


# ------------------------------------------------------------
# MobileDeepFilterNet Tiny
# ------------------------------------------------------------
class MobileDeepFilterNetTiny(nn.Module):
    """
    Tiny configuration of MobileDeepFilterNet.

    Designed for:
        - real-time speech enhancement
        - CPU/mobile inference

    Inputs
    ------
    magnitude_patch : [B, K_ctx, F]

    Outputs
    -------
    mask : [B, F]
        Spectral suppression mask.

    taps : [B, F, K_tap, 2]
        Complex deep filter coefficients.
    """

    def __init__(
        self,
        freq_bins=161,
        k_ctx=5,
        k_tap=3,
        encoder_channels=(16, 32, 48),
        gru_hidden=96,
        pooled_freq=32
    ):

        super().__init__()

        self.freq_bins = freq_bins
        self.k_ctx = k_ctx
        self.k_tap = k_tap
        self.pooled_freq = pooled_freq

        # ---------------- Encoder ----------------
        layers = []
        in_ch = 1

        for out_ch in encoder_channels:
            layers.append(DepthwiseSeparableConv(in_ch, out_ch))
            in_ch = out_ch

        self.encoder = nn.Sequential(*layers)

        # Pool frequency dimension to reduce GRU input size
        self.freq_pool = nn.AdaptiveAvgPool2d((k_ctx, pooled_freq))

        # ---------------- Temporal Modeling ----------------
        self.gru_input_size = encoder_channels[-1] * pooled_freq

        self.gru = nn.GRU(
            input_size=self.gru_input_size,
            hidden_size=gru_hidden,
            batch_first=True
        )

        # ---------------- Decoder Heads ----------------

        # Mask head
        self.mask_head = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden),
            nn.ReLU(),
            nn.Linear(gru_hidden, freq_bins),
            nn.Sigmoid()
        )

        # Deep filter taps head
        self.tap_head = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden),
            nn.ReLU(),
            nn.Linear(gru_hidden, freq_bins * k_tap * 2)
        )

    def forward(self, magnitude_patch):

        """
        Forward pass.

        Input
        -----
        magnitude_patch : [B, K_ctx, F]

        Returns
        -------
        mask : [B, F]

        taps : [B, F, K_tap, 2]
        """

        B, K, F = magnitude_patch.shape

        # Convert to 2D conv format
        x = magnitude_patch.unsqueeze(1)   # [B, 1, K_ctx, F]

        # Encoder
        x = self.encoder(x)                # [B, C, K_ctx, F]

        # Frequency pooling
        x = self.freq_pool(x)              # [B, C, K_ctx, pooled_freq]

        # Prepare for GRU
        B, C, T, Pf = x.shape
        x = x.permute(0, 2, 1, 3)          # [B, T, C, Pf]
        x = x.reshape(B, T, C * Pf)        # [B, T, features]

        # GRU temporal modeling
        out, h = self.gru(x)

        # Use final hidden state
        h = h.squeeze(0)                   # [B, hidden]

        # Decoder heads
        mask = self.mask_head(h)           # [B, F]

        taps = self.tap_head(h)
        taps = taps.view(B, self.freq_bins, self.k_tap, 2)

        return mask, taps


# ------------------------------------------------------------
# Smoke Test
# ------------------------------------------------------------
if __name__ == "__main__":

    B = 2
    K_ctx = 5
    F = 161

    x = torch.rand(B, K_ctx, F)

    model = MobileDeepFilterNetTiny()

    mask, taps = model(x)

    print("Mask shape:", mask.shape)
    print("Taps shape:", taps.shape)

In [ ]:
# Module 3 (Notebook Cell)
# Deep Filter Operator: apply hybrid mask + per-frequency complex deep-filter taps
#
# - Input shapes & conventions:
#     mask: [B, F]                (values in [0,1], spectral mask per frame)
#     taps: [B, F, K_tap, 2]      (complex taps: last dim = [real, imag])
#     complex_buffer: [B, K_tap, F, 2]  (past K_tap complex frames; index 0 = current X(f,t))
#
# - Output:
#     enhanced_spec: [B, F, 2]    (enhanced complex spectrum for current frame)
#
# Implementation notes:
# - Fully vectorized (no Python loops over frequency or taps) using PyTorch tensor ops.
# - Complex multiplication: (a+jb)*(x+jy) = (a*x - b*y) + j(a*y + b*x)
# - We permit three modes:
#     mode='hybrid'  -> mask * (sum_k taps * X(t-k))
#     mode='mask'    -> mask * X(current)  (mask-only baseline)
#     mode='filter'  -> sum_k taps * X(t-k) (filter-only baseline)
#
# - This operator expects tensors already on the correct device and dtype.
# - Comments and docstrings follow machine-learning best practices for clarity.

import torch
from typing import Literal

def apply_deep_filter_operator(
    mask: torch.Tensor,
    taps: torch.Tensor,
    complex_buffer: torch.Tensor,
    mode: Literal['hybrid', 'mask', 'filter'] = 'hybrid',
) -> torch.Tensor:
    """
    Apply deep-filter (and optional mask) to produce enhanced complex spectrum.

    Args:
        mask: [B, F] spectral mask (values in [0,1]).
        taps: [B, F, K_tap, 2] complex taps (real, imag).
        complex_buffer: [B, K_tap, F, 2] complex frames buffer where
                        complex_buffer[:, 0, :, :] == X(f,t) (current frame).
        mode: str, one of {'hybrid','mask','filter'}:
              - 'hybrid': enhanced = mask * sum_k taps * X(t-k)
              - 'mask'  : enhanced = mask * X(current)
              - 'filter': enhanced = sum_k taps * X(t-k)

    Returns:
        enhanced: [B, F, 2] enhanced complex spectrum (real, imag)
    """
    # Basic shape checks
    assert mask.dim() == 2, f"mask must be [B, F], got {mask.shape}"
    assert taps.dim() == 4 and taps.size(-1) == 2, f"taps must be [B, F, K_tap, 2], got {taps.shape}"
    assert complex_buffer.dim() == 4 and complex_buffer.size(-1) == 2, f"complex_buffer must be [B, K_tap, F, 2], got {complex_buffer.shape}"

    B_m, F_m = mask.shape
    B_t, F_t, K_tap_t, two_t = taps.shape
    B_b, K_b, F_b, two_b = complex_buffer.shape

    assert B_m == B_t == B_b, "Batch size mismatch between mask, taps, and buffer"
    assert F_m == F_t == F_b, "Frequency dimension mismatch"
    assert K_tap_t == K_b, "K_tap mismatch between taps and buffer"
    assert two_t == 2 and two_b == 2, "Last dim must be 2 (real,imag)"

    B = B_m
    F = F_m
    K_tap = K_tap_t

    device = mask.device
    dtype = mask.dtype

    # Mode: mask-only -> simple path: enhanced = mask * X(current)
    if mode == 'mask':
        # Current frame: complex_buffer[:, 0, :, :] -> [B, F, 2]
        current = complex_buffer[:, 0, :, :]  # [B, F, 2]
        # Apply mask (broadcast along complex channel)
        enhanced = current * mask.unsqueeze(-1)  # [B, F, 2]
        return enhanced

    # For filter or hybrid mode we compute the complex sum S(f) = sum_k taps[:,f,k] * X(t-k)
    # Rearrange taps to align with buffer: taps [B, F, K_tap, 2] -> taps_k [B, K_tap, F, 2]
    taps_k = taps.permute(0, 2, 1, 3).contiguous()  # [B, K_tap, F, 2]

    # Separate real/imag parts
    taps_real = taps_k[..., 0]  # [B, K, F]
    taps_imag = taps_k[..., 1]  # [B, K, F]
    buf_real = complex_buffer[..., 0]  # [B, K, F]
    buf_imag = complex_buffer[..., 1]  # [B, K, F]

    # Compute complex multiplication elementwise over k: (a+jb)*(x+jy)
    # real_k = a*x - b*y
    # imag_k = a*y + b*x
    real_k = taps_real * buf_real - taps_imag * buf_imag  # [B, K, F]
    imag_k = taps_real * buf_imag + taps_imag * buf_real  # [B, K, F]

    # Sum over K_tap dimension -> [B, F]
    real_sum = real_k.sum(dim=1)  # [B, F]
    imag_sum = imag_k.sum(dim=1)  # [B, F]

    # Stack to complex representation
    filtered = torch.stack([real_sum, imag_sum], dim=-1)  # [B, F, 2]

    if mode == 'filter':
        return filtered

    # Hybrid: apply mask to filtered output
    if mode == 'hybrid':
        enhanced = filtered * mask.unsqueeze(-1)  # broadcast mask over complex channels
        return enhanced

    raise ValueError(f"Unknown mode: {mode}. Supported: 'hybrid','mask','filter'.")


# ----------------------------- Smoke test --------------------------------
if __name__ == "__main__":
    # Basic shape smoke test to validate operator behavior.
    B = 2
    F = 161
    K_tap = 3

    # Random but deterministic inputs for testing
    torch.manual_seed(0)
    mask = torch.rand(B, F) * 0.9 + 0.05  # avoid zeros
    taps = torch.randn(B, F, K_tap, 2) * 0.1  # small filter taps
    # Construct complex_buffer such that index 0 is current, index 1 is previous, ...
    complex_buffer = torch.randn(B, K_tap, F, 2) * 0.5

    # Hybrid mode
    enhanced_hybrid = apply_deep_filter_operator(mask, taps, complex_buffer, mode='hybrid')
    print("hybrid enhanced shape:", enhanced_hybrid.shape)  # expected [B, F, 2]

    # Mask-only baseline
    enhanced_mask = apply_deep_filter_operator(mask, taps, complex_buffer, mode='mask')
    print("mask-only enhanced shape:", enhanced_mask.shape)  # expected [B, F, 2]

    # Filter-only baseline
    enhanced_filter = apply_deep_filter_operator(mask, taps, complex_buffer, mode='filter')
    print("filter-only enhanced shape:", enhanced_filter.shape)  # expected [B, F, 2]

    # Quick numeric check: hybrid should equal mask * filter (approx)
    diff = (enhanced_hybrid - (enhanced_filter * mask.unsqueeze(-1))).abs().max().item()
    print(f"hybrid vs mask*filter max-abs-diff: {diff:.6e}")

    print("Module 3 (Deep Filter Operator) cell executed successfully.")

In [ ]:
# ============================================================
# Module 4 (Notebook Cell)
# Real-Time Streaming Speech Enhancement
# ============================================================
# This module connects all previous components into a real-time
# streaming pipeline using sounddevice for audio I/O.
#
# Pipeline:
#
# microphone
#   ↓
# STFT
#   ↓
# magnitude patch buffer
#   ↓
# MobileDeepFilterNet
#   ↓
# Deep Filter Operator
#   ↓
# iSTFT
#   ↓
# speaker output
#
# Requirements:
# pip install sounddevice
#

import sounddevice as sd
import numpy as np
import torch

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SAMPLE_RATE = 16000
FRAME_SIZE = 320
HOP_SIZE = 160

K_CTX = 5
K_TAP = 3
FREQ_BINS = FRAME_SIZE // 2 + 1

DEVICE = torch.device("cpu")

# ------------------------------------------------------------
# Initialize Components
# ------------------------------------------------------------

stft_processor = STFTProcessor(
    sample_rate=SAMPLE_RATE,
    n_fft=FRAME_SIZE,
    hop_length=HOP_SIZE,
    win_length=FRAME_SIZE,
    center=False
)

model = MobileDeepFilterNetTiny(
    freq_bins=FREQ_BINS,
    k_ctx=K_CTX,
    k_tap=K_TAP
).to(DEVICE)

model.eval()

complex_buffer = ComplexFrameBuffer(
    K=K_TAP,
    freq_bins=FREQ_BINS
)

mag_history = []

# ------------------------------------------------------------
# Streaming Callback
# ------------------------------------------------------------

def audio_callback(indata, outdata, frames, time, status):

    global mag_history

    if status:
        print(status)

    # Convert numpy audio → torch tensor
    audio = torch.from_numpy(indata[:,0]).float().unsqueeze(0)

    # ---------------- STFT ----------------
    complex_spec = stft_processor.stft(audio)

    # [B,F,T,2] → take current frame
    complex_frame = complex_spec[:,:,0,:]

    # Magnitude
    magnitude = stft_processor.magnitude(complex_spec)[:,:,0]

    # ---------------- Buffers ----------------
    complex_buffer.append(complex_frame)

    mag_history.append(magnitude)

    if len(mag_history) > K_CTX:
        mag_history.pop(0)

    # If not enough frames yet → passthrough
    if len(mag_history) < K_CTX:

        outdata[:] = indata
        return

    mag_patch = torch.stack(mag_history, dim=1)

    # ---------------- Model ----------------
    with torch.no_grad():

        mask, taps = model(mag_patch)

    # ---------------- Deep Filter ----------------
    buf = complex_buffer.get_buffer()

    enhanced_spec = apply_deep_filter_operator(
        mask,
        taps,
        buf,
        mode="hybrid"
    )

    enhanced_spec = enhanced_spec.unsqueeze(2)

    # ---------------- iSTFT ----------------
    enhanced_audio = stft_processor.istft(
        enhanced_spec,
        length=frames
    )

    enhanced_audio = enhanced_audio.squeeze().cpu().numpy()

    outdata[:] = enhanced_audio.reshape(-1,1)

# ------------------------------------------------------------
# Start Streaming
# ------------------------------------------------------------

print("Starting real-time speech enhancement...")

stream = sd.Stream(
    samplerate=SAMPLE_RATE,
    blocksize=HOP_SIZE,
    channels=1,
    callback=audio_callback
)

with stream:
    sd.sleep(600000)

In [ ]:
# Module 5 (Notebook Cell)
# Training pipeline: dataset, batch preparation, training loop
#
# Copy/paste into the same notebook where Module 1..4 cells are defined.
# Requires: PyTorch, (optional) torchaudio or soundfile for audio loading.
#
# The implementation is written to be clear and educational while still
# usable as a real training script for MobileDeepFilterNetTiny.

import os
import math
import random
import glob
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Try to import torchaudio for audio loading; fallback to soundfile (sf).
try:
    import torchaudio
    _HAS_TORCHAUDIO = True
except Exception:
    _HAS_TORCHAUDIO = False
    try:
        import soundfile as sf
    except Exception:
        sf = None


# ----------------------------- Utilities ----------------------------------
def load_audio(path: str, target_sr: int = 16000) -> torch.Tensor:
    """
    Load audio from file and return mono float32 waveform tensor [samples].
    Uses torchaudio if available, else soundfile. Returns torch tensor.
    """
    if _HAS_TORCHAUDIO:
        wav, sr = torchaudio.load(path)
        wav = wav.mean(dim=0) if wav.ndim == 2 else wav  # to mono
        if sr != target_sr:
            wav = torchaudio.transforms.Resample(sr, target_sr)(wav.unsqueeze(0)).squeeze(0)
        return wav.float()
    else:
        if sf is None:
            raise RuntimeError("No audio backend available (install torchaudio or soundfile).")
        wav, sr = sf.read(path, dtype='float32')
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        # resample if needed using naive method (not high-quality)
        if sr != target_sr:
            import numpy as np
            duration = wav.shape[0] / sr
            new_len = int(duration * target_sr)
            wav = np.interp(
                np.linspace(0.0, duration, new_len, endpoint=False),
                np.linspace(0.0, duration, wav.shape[0], endpoint=False),
                wav
            ).astype('float32')
        return torch.from_numpy(wav).float()


def mix_clean_and_noise(clean: torch.Tensor, noise: torch.Tensor, snr_db: float) -> torch.Tensor:
    """
    Mix clean + noise at desired SNR (dB). Both inputs are 1D torch tensors.
    If noise shorter than clean, noise is looped/repeated. If longer, a random segment is chosen.
    Returns mixture (same length as clean).
    """
    clean = clean.float()
    noise = noise.float()

    L = clean.shape[0]
    if noise.shape[0] < L:
        # repeat noise to match length
        reps = math.ceil(L / noise.shape[0])
        noise = noise.repeat(reps)[:L]
    elif noise.shape[0] > L:
        # pick random segment
        start = random.randint(0, noise.shape[0] - L)
        noise = noise[start:start+L]

    # compute power
    eps = 1e-9
    clean_power = (clean ** 2).mean()
    noise_power = (noise ** 2).mean() + eps

    # desired noise scale
    snr_linear = 10 ** (snr_db / 10.0)
    scale = torch.sqrt(clean_power / (snr_linear * noise_power + eps))

    noisy = clean + noise * scale
    return noisy


# ----------------------------- Dataset ------------------------------------
class SpeechNoiseDataset(Dataset):
    """
    Dataset for on-the-fly mixing of clean speech and noise.
    Expects lists of file paths for clean and noise directories.

    For quick testing, if file lists are None, a synthetic dataset will be used.
    """

    def __init__(
        self,
        clean_files: Optional[List[str]] = None,
        noise_files: Optional[List[str]] = None,
        sample_rate: int = 16000,
        segment_len_seconds: float = 4.0,
        snr_range: Tuple[float, float] = (-5.0, 15.0),
        preload: bool = False
    ):
        self.clean_files = clean_files or []
        self.noise_files = noise_files or []
        self.sample_rate = sample_rate
        self.segment_len = int(segment_len_seconds * sample_rate)
        self.snr_range = snr_range
        self.preload = preload

        # If no files provided, synthetic random dataset will be used
        self.synthetic = (len(self.clean_files) == 0 or len(self.noise_files) == 0)

        if not self.synthetic and self.preload:
            # Preload audio into memory for speed
            self.clean_wavs = [load_audio(p, target_sr=sample_rate) for p in self.clean_files]
            self.noise_wavs = [load_audio(p, target_sr=sample_rate) for p in self.noise_files]
        else:
            self.clean_wavs = None
            self.noise_wavs = None

        # Define dataset length
        self._len = max(1000, len(self.clean_files)) if self.synthetic else len(self.clean_files)

    def __len__(self):
        return self._len

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
          clean_segment: [segment_len] float tensor
          noise_segment: [segment_len] float tensor (random segment)
        For synthetic mode, returns random noise and sine-like clean signal.
        """
        if self.synthetic:
            # Create synthetic clean speech: mixture of chirps/sines with random amplitude envelope
            t = torch.linspace(0, self.segment_len / self.sample_rate, steps=self.segment_len)
            freqs = torch.tensor([random.uniform(100, 3000), random.uniform(100, 4000)])
            clean = 0.5 * torch.sin(2 * math.pi * freqs[0] * t) + 0.3 * torch.sin(2 * math.pi * freqs[1] * t)
            # amplitude envelope
            env = torch.rand(self.segment_len).cumsum(0)
            env = env / env.max()
            clean = clean * (0.5 + 0.5 * env)
            # noise: gaussian
            noise = 0.1 * torch.randn(self.segment_len)
            return clean, noise
        else:
            # Load clean file (preloaded or on-the-fly)
            if self.clean_wavs is not None:
                wav = self.clean_wavs[idx % len(self.clean_wavs)]
            else:
                wav = load_audio(self.clean_files[idx % len(self.clean_files)], target_sr=self.sample_rate)

            # If wav shorter than segment, loop; else random crop
            if wav.shape[0] < self.segment_len:
                reps = math.ceil(self.segment_len / wav.shape[0])
                clean = wav.repeat(reps)[:self.segment_len]
            else:
                start = random.randint(0, wav.shape[0] - self.segment_len)
                clean = wav[start:start+self.segment_len]

            # Sample a noise file and pick a random segment
            noise_path = random.choice(self.noise_files)
            noise_wav = load_audio(noise_path, target_sr=self.sample_rate)
            if noise_wav.shape[0] < self.segment_len:
                reps = math.ceil(self.segment_len / noise_wav.shape[0])
                noise = noise_wav.repeat(reps)[:self.segment_len]
            else:
                start = random.randint(0, noise_wav.shape[0] - self.segment_len)
                noise = noise_wav[start:start+self.segment_len]

            return clean, noise


# ------------------------- Batch preparation helpers -----------------------
def build_mag_patches_and_buffers(
    noisy_wave: torch.Tensor,
    stft_proc: 'STFTProcessor',
    k_ctx: int,
    k_tap: int
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Given a noisy waveform [B, samples], compute its complex STFT and produce:
      - mag_patches: [B, T, K_ctx, F]  (we will reshape to [B*T, K_ctx, F])
      - complex_buffers: [B, T, K_tap, F, 2] (for each frame t, K_tap past frames)
    Notes:
      - We produce patches for each time frame t where t corresponds to STFT frame index.
      - Use zero-padding for contexts where t-k < 0 (initial frames).
    Returns:
      mag_patches, complex_buffers, complex_noisy_spec (for debugging): shapes described above
    """
    # noisy_wave: [B, samples]
    assert noisy_wave.dim() == 2
    B = noisy_wave.size(0)

    # STFT: complex_spec_noisy [B, F, T, 2]
    complex_spec_noisy = stft_proc.stft(noisy_wave)  # [B, F, T, 2]
    Bc, F, T, _ = complex_spec_noisy.shape
    assert Bc == B

    # magnitude: [B, F, T]
    mag = stft_proc.magnitude(complex_spec_noisy)  # [B, F, T]

    # We'll create mag_patches per frame t: for causal context t-K_ctx+1 .. t
    # mag_patches_all shape [B, T, K_ctx, F]
    mag_patches_all = torch.zeros(B, T, k_ctx, F, device=noisy_wave.device, dtype=mag.dtype)
    # complex_buffers shape [B, T, K_tap, F, 2]
    complex_buffers = torch.zeros(B, T, k_tap, F, 2, device=noisy_wave.device, dtype=complex_spec_noisy.dtype)

    # For each frame t, collect context and buffer (vectorized via loops over t; OK for moderate T)
    # Note: could be optimized by fancy indexing; kept simple for clarity.
    for t in range(T):
        # context indices
        ctx_indices = [t - (k_ctx - 1) + i for i in range(k_ctx)]  # from t-K_ctx+1 .. t
        # for negative indices, use zeros (already zero-initialized)
        for i, ti in enumerate(ctx_indices):
            if 0 <= ti < T:
                # mag[:,:,ti] -> [B, F]
                mag_patches_all[:, t, i, :] = mag[:, :, ti]
        # buffer taps: k=0..K_tap-1 -> X(t-k)
        for k in range(k_tap):
            ti = t - k
            if 0 <= ti < T:
                # complex_spec_noisy[:, :, ti, :] shape [B, F, 2]
                complex_buffers[:, t, k, :, :] = complex_spec_noisy[:, :, ti, :]

    # Return shapes:
    # mag_patches_all: [B, T, K_ctx, F] -> reshape to [B*T, K_ctx, F] for model batching
    mag_patches_flat = mag_patches_all.view(B * T, k_ctx, F)
    # complex_buffers_flat: [B*T, K_tap, F, 2]
    complex_buffers_flat = complex_buffers.view(B * T, k_tap, F, 2)

    return mag_patches_flat, complex_buffers_flat, complex_spec_noisy


# ----------------------------- Training Loop ------------------------------
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    stft_proc: 'STFTProcessor',
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epoch: int = 0,
    grad_clip: float = 1.0,
    sr: int = 16000,
    k_ctx: int = 5,
    k_tap: int = 3,
    verbose: bool = True
):
    """
    Single epoch training loop.

    For each batch:
      - mix clean+noise at random SNR
      - compute mag_patches and complex buffers
      - forward model on mag_patches (vectorized all frames)
      - apply deep-filter on each frame (vectorized)
      - reconstruct enhanced waveform with iSTFT
      - compute SI-SDR loss and backprop
    """
    model.train()
    total_loss = 0.0
    n_batches = 0

    for batch_idx, (clean_seg, noise_seg) in enumerate(dataloader):
        # Move to device
        clean_seg = clean_seg.to(device)  # [B, L]
        noise_seg = noise_seg.to(device)

        B, L = clean_seg.shape

        # Choose random SNR per example in this batch
        snrs = torch.empty(B).uniform_( -5.0, 15.0 ).tolist()  # example SNR range

        noisy_batch = torch.zeros_like(clean_seg)
        for i in range(B):
            noisy_batch[i] = mix_clean_and_noise(clean_seg[i], noise_seg[i], snrs[i])

        # Build mag_patches and complex buffers (vectorized)
        mag_patches_flat, complex_buffers_flat, complex_noisy_spec = build_mag_patches_and_buffers(
            noisy_batch, stft_proc, k_ctx=k_ctx, k_tap=k_tap
        )
        # mag_patches_flat: [B*T, K_ctx, F]
        # complex_buffers_flat: [B*T, K_tap, F, 2]

        # Run model in minibatches if too large to fit into memory (safety)
        # We'll split the large (B*T) dimension into chunks of max_chunk
        max_chunk = 1024  # tune based on GPU memory
        n_total_patches = mag_patches_flat.shape[0]
        masks_list = []
        taps_list = []

        with torch.set_grad_enabled(True):
            start = 0
            while start < n_total_patches:
                end = min(start + max_chunk, n_total_patches)
                patch_chunk = mag_patches_flat[start:end].to(device)  # [C, K_ctx, F]
                # Model expects [B_patch, K_ctx, F]
                mask_chunk, taps_chunk = model(patch_chunk)
                masks_list.append(mask_chunk)
                taps_list.append(taps_chunk)
                start = end

            # Concatenate
            masks_flat = torch.cat(masks_list, dim=0)  # [B*T, F]
            taps_flat = torch.cat(taps_list, dim=0)    # [B*T, F, K_tap, 2]

            # Apply deep filter operator (vectorized)
            enhanced_frames_flat = apply_deep_filter_operator(
                masks_flat,
                taps_flat,
                complex_buffers_flat.to(device),
                mode='hybrid'
            )  # [B*T, F, 2]

            # Reshape enhanced frames into [B, F, T, 2] to perform iSTFT
            # Note: we built mag_patches_flat from shape [B, T, K_ctx, F] -> flat.
            # Recover T from complex_noisy_spec.shape
            _, F_spec, T_spec, _ = complex_noisy_spec.shape
            enhanced_frames = enhanced_frames_flat.view(B, T_spec, F_spec, 2).permute(0,2,1,3).contiguous()  # [B, F, T, 2]

            # Inverse STFT to waveform
            # For torch.istft with center=False streaming semantics, ensure length matches L
            enhanced_wav = stft_proc.istft(enhanced_frames, length=L)  # [B, L]

            # Compute SI-SDR loss
            loss = si_sdr_loss(enhanced_wav, clean_seg.to(device))
            # Optional spectral MSE auxiliary loss:
            # compute clean spec and compare magnitudes (not necessary but helpful)
            # clean_spec = stft_proc.stft(clean_seg.to(device))
            # clean_mag = stft_proc.magnitude(clean_spec)
            # enhanced_mag = stft_proc.magnitude(enhanced_frames)
            # spec_loss = nn.functional.mse_loss(enhanced_mag, clean_mag)
            # combined_loss = loss + 0.1 * spec_loss

            optimizer.zero_grad()
            loss.backward()
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        if verbose and (batch_idx % 10 == 0):
            print(f"Epoch {epoch} Batch {batch_idx} Loss {loss.item():.4f}")

    avg_loss = total_loss / max(1, n_batches)
    return avg_loss


# ----------------------------- Top-level trainer ---------------------------
def train_model(
    model: nn.Module,
    dataset: Dataset,
    stft_proc: 'STFTProcessor',
    device: torch.device = torch.device('cpu'),
    epochs: int = 10,
    batch_size: int = 2,
    lr: float = 1e-3,
    save_dir: str = './checkpoints',
    k_ctx: int = 5,
    k_tap: int = 3,
):
    """
    High-level training function that trains model on dataset.
    Saves checkpoints at each epoch.
    """
    os.makedirs(save_dir, exist_ok=True)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False, drop_last=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    model.to(device)

    for epoch in range(1, epochs + 1):
        avg_loss = train_one_epoch(
            model, dataloader, stft_proc, optimizer, device,
            epoch=epoch, grad_clip=1.0, sr=stft_proc.sample_rate,
            k_ctx=k_ctx, k_tap=k_tap
        )
        print(f"Epoch {epoch} Average Loss: {avg_loss:.4f}")

        # Save checkpoint
        ckpt_path = os.path.join(save_dir, f'mobile_deepfilternet_epoch{epoch}.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'avg_loss': avg_loss
        }, ckpt_path)
        print(f"Saved checkpoint: {ckpt_path}")

        scheduler.step()


# ----------------------------- Quick demo ---------------------------------
if __name__ == "__main__":
    # Quick demo training run using synthetic dataset for a few iterations
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Using device:", device)

    # Instantiate components (reuse values from previous modules)
    SR = 16000
    FRAME = 320
    HOP = 160
    K_CTX = 5
    K_TAP = 3

    stft_proc = STFTProcessor(sample_rate=SR, n_fft=FRAME, hop_length=HOP, win_length=FRAME, center=False, device=device)
    model = MobileDeepFilterNetTiny(freq_bins=FRAME//2+1, k_ctx=K_CTX, k_tap=K_TAP).to(device)

    # Synthetic dataset (quick test)
    ds = SpeechNoiseDataset(clean_files=None, noise_files=None, sample_rate=SR, segment_len_seconds=2.0, preload=False)
    # Train for 2 epochs with small batch
    train_model(model, ds, stft_proc, device=device, epochs=2, batch_size=2, lr=1e-3, save_dir='./checkpoints_demo', k_ctx=K_CTX, k_tap=K_TAP)

    print("Demo training completed.")